# Student Notebook 04 — Explanation (Layer 4)

TreeSHAP attribution: which features pushed each film's
probability up, which pulled it down. Works for tree-based
models (xgboost, random forest). For linear models we fall
back to coefficients; for SVM we fall back to permutation
importance.

**Run order:** notebook 01 must run first. Notebooks 02 and
03 are not required for this notebook.

**What you'll get out:** global feature importance bar chart,
per-film SHAP for one example film of your choice.

## CONFIG — edit me

The "which film to inspect" knob is the interesting one. By
default we pick the film with the highest predicted
probability so the SHAP signal is strongest, but you can
substitute any imdb_id from the cal set.

In [ ]:
CONFIG = {
    # MUST match the run_name you used in 01.
    "run_name": "team_baseline_rf",

    "top_k_global":   20,        # how many features to plot in the global bar chart
    "top_k_per_film": 5,         # how many positive + negative contributors per film
    "example_imdb_id": "auto",   # "auto" picks highest-probability film, or paste any imdb_id
}

## Imports + load

In [ ]:
# --- Paths (works from any working directory) ---
from pathlib import Path

def _find_project_root() -> Path:
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "docs" / "PROJECT_CONTEXT.md").is_file():
            return cand
    raise RuntimeError(f"Could not locate project root from {Path.cwd()!s}")

PROJECT_ROOT = _find_project_root()
DATA = PROJECT_ROOT / "data" / "processed"
STUDENT = DATA / "student" / CONFIG["run_name"]
STUDENT.mkdir(parents=True, exist_ok=True)
print(f"Project root:  {PROJECT_ROOT}")
print(f"Run name:      {CONFIG['run_name']!r}")
print(f"Artifacts go:  {STUDENT.relative_to(PROJECT_ROOT)}")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print("WARNING: shap not installed. pip install shap")

bundle = joblib.load(STUDENT / "student_model.joblib")
feat = pd.read_parquet(DATA / "features.parquet").reset_index()
master = pd.read_parquet(DATA / "films_joined.parquet")

df_cal = feat[feat["split"] == "cal"].reset_index(drop=True)
feat_cols = bundle["feature_columns"]

X_cal = df_cal[feat_cols].fillna(0).values
ids = df_cal["imdb_id"].astype(str).tolist()
proba = bundle["model"].predict_proba(X_cal)[:, 1]

family = bundle["config"]["model_family"]
print(f"Loaded {family} model on target {bundle['config']['target']!r}; "
      f"{len(df_cal)} cal films")

## Compute attributions

Three branches based on model family:

- **Tree models (xgboost, random_forest):** TreeSHAP, exact +
  fast. Returns per-film signed contributions.
- **Logistic:** coefficients × scaled features. Linear, exact.
- **SVM:** permutation importance (slow but model-agnostic).

In [ ]:
def attribute(model_pipeline, X, family):
    scaler = model_pipeline.named_steps["scaler"]
    clf = model_pipeline.named_steps["model"]
    X_scaled = scaler.transform(X)

    if family in ("xgboost", "random_forest"):
        if not HAS_SHAP:
            raise RuntimeError("shap not installed")
        explainer = shap.TreeExplainer(clf)
        raw = explainer.shap_values(X_scaled)
        arr = np.asarray(raw)
        if arr.ndim == 3:
            arr = arr[:, :, 1]  # positive class
        return arr, "TreeSHAP"

    if family == "logistic":
        # Per-film contribution = coefficient × scaled feature value.
        coefs = clf.coef_.ravel()
        contributions = X_scaled * coefs[np.newaxis, :]
        return contributions, "linear coefficients × x"

    if family == "svm_rbf":
        from sklearn.inspection import permutation_importance
        # Permutation importance is global (not per-film); we tile.
        # For a per-film story on SVM, KernelSHAP would work but takes hours.
        pi = permutation_importance(model_pipeline, X, model_pipeline.predict(X),
                                     n_repeats=5, random_state=42, n_jobs=-1)
        # Fake "per-film" array by replicating the global mean signed importance.
        return np.tile(pi.importances_mean, (len(X), 1)), "permutation importance (global only)"

    raise ValueError(f"Unsupported family for explanation: {family!r}")

contribs, method = attribute(bundle["model"], X_cal, family)
print(f"Attribution method: {method}")
print(f"Per-film contribution matrix shape: {contribs.shape}")

## Global ranking — top features by mean |contribution|

In [ ]:
mean_abs = np.mean(np.abs(contribs), axis=0)
mean_signed = np.mean(contribs, axis=0)
ranking = pd.DataFrame({
    "feature":          feat_cols,
    "mean_abs":         mean_abs,
    "mean_signed":      mean_signed,
}).sort_values("mean_abs", ascending=False).reset_index(drop=True)
print(ranking.head(15).to_string(index=False))

In [ ]:
K = CONFIG["top_k_global"]
top = ranking.head(K).iloc[::-1]  # reverse for top-down bars
colors = ["#2c7fb8" if s >= 0 else "#d7301f" for s in top["mean_signed"]]
fig, ax = plt.subplots(figsize=(8, K * 0.3 + 1))
ax.barh(top["feature"], top["mean_abs"], color=colors)
ax.set_xlabel("Mean |contribution|")
ax.set_title(f"Top-{K} feature importance — {family} on {bundle['config']['target']}")
ax.grid(axis="x", alpha=0.3)
plt.show()
print("Blue = pushes probability UP on average; Red = pulls DOWN on average")

## Per-film attribution

Pick the film with the highest predicted probability (or
whatever imdb_id you put in CONFIG). The bar chart shows
which features pushed *this specific film*'s probability up
or down.

In [ ]:
if CONFIG["example_imdb_id"] == "auto":
    example_idx = int(np.argmax(proba))
    example_id = ids[example_idx]
else:
    example_id = CONFIG["example_imdb_id"]
    example_idx = ids.index(example_id)

movie = master.loc[master["imdb_id"] == example_id, "movie_name"].iloc[0]
print(f"Inspecting: {movie} ({example_id})")
print(f"  Predicted probability: {proba[example_idx]:.3f}")

film_contribs = pd.DataFrame({
    "feature":      feat_cols,
    "contribution": contribs[example_idx],
})
# Top K positive and top K negative, by absolute contribution.
top_pos = film_contribs[film_contribs["contribution"] > 0].nlargest(CONFIG["top_k_per_film"], "contribution")
top_neg = film_contribs[film_contribs["contribution"] < 0].nsmallest(CONFIG["top_k_per_film"], "contribution")
show = pd.concat([top_pos, top_neg]).iloc[::-1]
colors = ["#2c7fb8" if v >= 0 else "#d7301f" for v in show["contribution"]]

fig, ax = plt.subplots(figsize=(8, len(show) * 0.4 + 1))
ax.barh(show["feature"], show["contribution"], color=colors)
ax.axvline(0, color="black", linewidth=0.5)
ax.set_xlabel("Per-film contribution to predicted log-odds")
ax.set_title(f"{movie} — top contributors")
ax.grid(axis="x", alpha=0.3)
plt.show()

## Compare to your teammates

Pick your favorite test film, paste its imdb_id into
CONFIG, and re-run the per-film cell. Each teammate
inspecting a different film will help you see whether the
system is reading the screenplay's actual content or just
the metadata genre tag.

Reference (rigorous Phase 7): on the cal set, the top-5
SHAP features for ``roi_gt_2`` are release_year_parsed,
genre_Horror, network_lead_role_count, genre_Action,
genre_Romance. Your top-5 should overlap heavily.

TreeSHAP doesn't apply to SVM; if your team is comparing
SVM vs xgboost on layer 1, only the xgboost run will produce
a meaningful per-film SHAP.